<a href="https://colab.research.google.com/github/Khaledsaads/gemma-7b-sft-dpo/blob/main/gemma_7b_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -- no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-nj597sm5/unsloth_b2dd3196d9454bfa9c6656d3966e261d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-nj597sm5/unsloth_b2dd3196d9454bfa9c6656d3966e261d
  Resolved https://github.com/unslothai/unsloth.git to commit 1492952a0924c4b1d8ca5994257c7c2abdf87934
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement no-deps (from versions: none)
ERROR: No matching distribution found for no-deps


In [2]:
! pip install trl

In [3]:
!pip install -U torchao

In [4]:
! pip install transformers

In [5]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

In [6]:
from trl import DPOTrainer, DPOConfig
from peft import PeftModel
from transformers import AutoTokenizer
from datasets import load_dataset
from unsloth import FastLanguageModel

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1684: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
dataset = load_dataset("argilla/dpo-mix-7k")

In [8]:
from unsloth.chat_templates import get_chat_template
tokenizer = AutoTokenizer.from_pretrained("Khaledsaad1/gemma-7b-deita-10k-qlora")
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma")

In [9]:
for key in dataset["train"][0].keys():
  print(f"{key}: {dataset["train"][0][key]}")

dataset: argilla/distilabel-intel-orca-dpo-pairs
chosen: [{'content': 'Q:Question: how old julio cesar chavez when he fought de la hoya I found the following answer on Google: He holds records for most successful consecutive defenses of world titles (27), most title fights (37), most title-fight victories (31) and he is after Joe Louis with (23) for most title defenses won by knockout (21). Is that a correct answer? Yes or no.\nA:', 'role': 'user'}, {'content': " Yes, the information you found on Google is correct. Julio César Chávez holds several records related to world title defenses and victories, and he is considered one of the greatest boxers in history. Here is a detailed answer to your question:\n\nJulio César Chávez was born on July 12, 1962, in Ciudad Obregón, Sonora, Mexico. He began boxing at a young age and quickly made a name for himself in the sport, winning his first world title in 1984 when he defeated Mario Miranda for the WBC super featherweight championship.\n\nOver

In [10]:
def apply_dpo_template(example):
  if all(k in example.keys() for k in ("chosen", "rejected")):
    prompt_text = example["chosen"][:-1]
    chosen_text = example["chosen"][-1:]
    rejected_text = example["rejected"][-1:]
    example["text_prompt"] = tokenizer.apply_chat_template(prompt_text, tokenize= False)
    example["text_chosen"] = tokenizer.apply_chat_template(chosen_text, tokenize= False)
    example["text_rejected"] = tokenizer.apply_chat_template(rejected_text, tokenize= False)
  return example

In [11]:
dataset["train"]

Dataset({
    features: ['dataset', 'chosen', 'rejected', 'chosen_rating', 'rejected_rating'],
    num_rows: 6750
})

In [12]:
dataset['test']

Dataset({
    features: ['dataset', 'chosen', 'rejected', 'chosen_rating', 'rejected_rating'],
    num_rows: 750
})

In [13]:
columns_names = (dataset['train'].features)
dataset_template = dataset.map(apply_dpo_template, remove_columns = columns_names,desc="Formatting comparisons with prompt template")
for split in ["train", "test"]:
  dataset_template[split] = dataset_template[split].rename_columns(
      {"text_prompt": "prompt", "text_chosen":"chosen", "text_rejected": "rejected"}
  )

In [14]:
dataset_template["train"].features

{'prompt': Value('string'),
 'chosen': Value('string'),
 'rejected': Value('string')}

In [15]:
dataset_template["test"].features

{'prompt': Value('string'),
 'chosen': Value('string'),
 'rejected': Value('string')}

In [16]:
dataset_template

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 6750
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 750
    })
})

In [17]:
for key in dataset_template["train"][0].keys():
  print(f"{key}: {dataset_template["train"][0][key]}")

prompt: <bos><start_of_turn>user
Q:Question: how old julio cesar chavez when he fought de la hoya I found the following answer on Google: He holds records for most successful consecutive defenses of world titles (27), most title fights (37), most title-fight victories (31) and he is after Joe Louis with (23) for most title defenses won by knockout (21). Is that a correct answer? Yes or no.
A:<end_of_turn>

chosen: <bos><start_of_turn>model
Yes, the information you found on Google is correct. Julio César Chávez holds several records related to world title defenses and victories, and he is considered one of the greatest boxers in history. Here is a detailed answer to your question:

Julio César Chávez was born on July 12, 1962, in Ciudad Obregón, Sonora, Mexico. He began boxing at a young age and quickly made a name for himself in the sport, winning his first world title in 1984 when he defeated Mario Miranda for the WBC super featherweight championship.

Over the course of his career, C

In [18]:
import numpy as np
import json
for feature in dataset_template["train"].features:
  lengths = [len(tokenizer(json.dumps(text))["input_ids"]) for text in dataset_template['train'][feature]]
  print(f"{feature} p99: {np.percentile(lengths, 99)}")
  print(f"{feature} p95: {np.percentile(lengths, 95)}")
  print(f"{feature} p90: {np.percentile(lengths, 90)}")

prompt p99: 2161.040000000001
prompt p95: 1451.0
prompt p90: 1112.0
chosen p99: 1017.590000000002
chosen p95: 759.0999999999985
chosen p90: 621.1000000000004
rejected p99: 1192.320000000007
rejected p95: 802.0
rejected p90: 640.0


In [19]:
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Khaledsaad1/gemma-7b-deita-10k-qlora",
    dtype =None,
    max_seq_length = 2048,
    use_gradient_checkpointing=False,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.10.8: Fast Gemma patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-7b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [20]:
model.print_trainable_parameters()

trainable params: 50,003,968 || all params: 8,587,684,864 || trainable%: 0.5823


In [21]:
print("Adapters before DPO:", model.peft_config.keys())

Adapters before DPO: dict_keys(['default'])


In [22]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma")

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer__cu9657_/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/_unsloth_sentencepiece_temp/tokenizer__cu9657_.


In [23]:
from unsloth import FastLanguageModel
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from transformers import EarlyStoppingCallback

In [24]:
trainer_args = DPOConfig(
    output_dir="./gemma_7B_dpo_model",
    num_train_epochs = 2,
    per_device_train_batch_size =1,
    per_device_eval_batch_size =16,
    gradient_accumulation_steps=16,
    eval_strategy = "steps",
    save_strategy = "steps",
    save_steps = 80,
    eval_steps = 80,
    logging_steps =40,
    warmup_steps = 40,
    beta = 0.05,
    learning_rate = 2e-5,
    optim = "paged_adamw_8bit",
    lr_scheduler_type = "cosine",
    weight_decay = 0.05,
    gradient_checkpointing =False,
    bf16=True,
    fp16= False,
    max_length = 2048,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better  = False,
    torch_compile=False
)

In [25]:
from transformers import EarlyStoppingCallback
trainer = DPOTrainer(
    model,
    ref_model = None,
    train_dataset = dataset_template["train"],
    eval_dataset = dataset_template["test"],
    args = trainer_args,
    processing_class = tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

Tokenizing train dataset (num_proc=6):   0%|          | 0/6750 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=6):   0%|          | 0/6750 [00:00<?, ? example…

Tokenizing eval dataset (num_proc=6):   0%|          | 0/750 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset (num_proc=6):   0%|          | 0/750 [00:00<?, ? examples/…

In [26]:
trainer.train(resume_from_checkpoint="/content/gemma_7B_dpo_model/checkpoint-480")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,709 | Num Epochs = 2 | Total steps = 840
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 50,003,968 of 8,637,688,832 (0.58% trained)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
560,0.631789,0.582193,0.948169,1619205.000000,210.950698,212.480805,0.668231,0.242348,-1.022412,0.718972,1.264760,-2239.169241,-2299.930233
640,0.502086,0.540391,1.011780,3241797.000000,218.107279,219.566620,0.655136,-2.556625,-5.176104,0.758865,2.619479,-2295.148718,-2383.004088
720,0.571136,0.460994,1.022097,4895448.000000,223.949314,225.381781,0.666038,-0.782890,-2.916063,0.782506,2.133172,-2259.674026,-2337.803264
800,0.480923,0.465995,0.936030,6513373.000000,225.116599,226.717860,0.669123,-0.800746,-2.892558,0.783540,2.091812,-2260.031125,-2337.333158
840,0.491203,0.455060,0.932338,7295367.000000,225.120839,226.755721,0.669100,-0.874771,-2.999195,0.800827,2.124424,-2261.511641,-2339.465893


Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-560/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-640/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-640.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-720/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-720.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-800/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-800.
Unsloth: Restored added_tokens_decoder metadata in ./gemma_7B_dpo_model/checkpoint-840/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./gemma_7B_dpo_model/checkpoint-840.


TrainOutput(global_step=840, training_loss=0.24137933821905228, metrics={'train_runtime': 5355.7268, 'train_samples_per_second': 2.505, 'train_steps_per_second': 0.157, 'total_flos': 8.938522117815583e+17, 'train_loss': 0.24137933821905228, 'epoch': 2.0})

In [27]:
sdfsd

NameError: name 'sdfsd' is not defined

In [29]:
path = "/content/gemma_7B_dpo_model/checkpoint-840/trainer_state.json"

with open(path, "r") as f:
    state = json.load(f)

for entry in state["log_history"]:
    if "learning_rate" in entry:
        print(f"Step: {entry.get('step')}, LR: {entry['learning_rate']}")

Step: 40, LR: 0.000195
Step: 80, LR: 0.00019882950400358694
Step: 120, LR: 0.00019522626868413956
Step: 160, LR: 0.0001892782465918221
Step: 200, LR: 0.000181131897770901
Step: 240, LR: 0.00017098781236552903
Step: 280, LR: 0.00015909577142467573
Step: 320, LR: 0.00014574859646372605
Step: 360, LR: 0.0001312749392269526
Step: 400, LR: 0.00011603118919083913
Step: 440, LR: 0.0001003926980723806
Step: 480, LR: 8.474453742408992e-05
Step: 520, LR: 6.947201689439565e-05
Step: 560, LR: 5.495119662532857e-05
Step: 600, LR: 4.1539627403752666e-05
Step: 640, LR: 2.956754657434937e-05
Step: 680, LR: 1.9329746500197023e-05
Step: 720, LR: 1.1078315796431571e-05
Step: 760, LR: 5.0164320719082285e-06
Step: 800, LR: 1.2933590221634428e-06
Step: 840, LR: 7.710618529443991e-10


In [30]:
from transformers import TrainingArguments
import torch

path = "/content/gemma_7B_dpo_model/checkpoint-840/training_args.bin"

args = torch.load(path, map_location="cpu", weights_only=False)

print("Configured LR:", args.learning_rate)

Configured LR: 2e-05


In [31]:
from huggingface_hub import login
login()

In [32]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = "Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint"
api.create_repo(
    repo_id = repo_id,
    repo_type = "model" ,
    exist_ok =True

    )
api.upload_folder(
    repo_id = repo_id,
    repo_type = "model",
    folder_path = "/content/gemma_7B_dpo_model/checkpoint-840")

CommitInfo(commit_url='https://huggingface.co/Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint/commit/0b634327332c7a1a39c05d4f4e9c12e825cda0cc', commit_message='Upload folder using huggingface_hub', commit_description='', oid='0b634327332c7a1a39c05d4f4e9c12e825cda0cc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint', endpoint='https://huggingface.co', repo_type='model', repo_id='Khaledsaad1/argilla-dpo-mix-7k-DPO-840-checkpoint'), pr_revision=None, pr_num=None)